In [0]:
# bronze 테이블 가져오기
bronze_df = spark.read.table("training.sjh.amazon_delivery_bronze")

In [0]:
bronze_df.show()

+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+-----------+--------------+-------------+-----------+
|     Order_ID|Agent_Age|Agent_Rating|Store_Latitude|Store_Longitude|Drop_Latitude|Drop_Longitude|Order_Date|Order_Time|        Pickup_Time|   Weather|Traffic|    Vehicle|          Area|Delivery_Time|   Category|
+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+-----------+--------------+-------------+-----------+
|ialx566343618|       37|         4.9|     22.745049|      75.892471|    22.765049|     75.912471|2022-03-19|  11:30:00|2026-08-30 11:45:00|     Sunny|  High |motorcycle |        Urban |          120|   Clothing|
|akqg208421122|       34|         4.5|     12.913041|      77.683237|    13.043041|     77.813237|2022-03-25|  19:45:00|2026-08-30 19:50:00|    Stor

In [0]:
# bronze_df 를 정제하여 silver_df 생성
# Agent_Rating이 null이거나 정상 범위 아닌 경우는 제외
# Order_Time이 'NaN ' 문자열인 행 제거
# 문자열 공백 정제

import pyspark.sql.functions as F

silver_df = (
    spark.table("training.sjh.amazon_delivery_bronze")
    # 결측치 및 이상치 행 완전 제거
    .filter(F.col("Agent_Rating").isNotNull())
    .filter(F.col("Agent_Rating") <= 5.0)
    .filter((F.col("Order_Time").isNotNull()) & (F.trim(F.col("Order_Time")) != "NaN"))
    
    # 문자열 공백 정제
    .withColumn("Weather", F.trim(F.col("Weather")))
    .withColumn("Traffic", F.trim(F.col("Traffic")))
    .withColumn("Vehicle", F.trim(F.col("Vehicle")))
    .withColumn("Area", F.trim(F.col("Area")))
)

In [0]:
silver_df.show()

+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+----------+-------------+-------------+-----------+
|     Order_ID|Agent_Age|Agent_Rating|Store_Latitude|Store_Longitude|Drop_Latitude|Drop_Longitude|Order_Date|Order_Time|        Pickup_Time|   Weather|Traffic|   Vehicle|         Area|Delivery_Time|   Category|
+-------------+---------+------------+--------------+---------------+-------------+--------------+----------+----------+-------------------+----------+-------+----------+-------------+-------------+-----------+
|ialx566343618|       37|         4.9|     22.745049|      75.892471|    22.765049|     75.912471|2022-03-19|  11:30:00|2026-08-30 11:45:00|     Sunny|   High|motorcycle|        Urban|          120|   Clothing|
|akqg208421122|       34|         4.5|     12.913041|      77.683237|    13.043041|     77.813237|2022-03-25|  19:45:00|2026-08-30 19:50:00|    Stormy|    J

In [0]:
# silver_df 테이블 생성
silver_df.write.mode("overwrite").saveAsTable("training.sjh.amazon_delivery_silver")

In [0]:
%sql

-- silver_df 테이블 조회
SELECT * FROM training.sjh.amazon_delivery_silver

Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,2026-08-30T11:45:00.000Z,Sunny,High,motorcycle,Urban,120,Clothing
akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,2026-08-30T19:50:00.000Z,Stormy,Jam,scooter,Metropolitian,165,Electronics
njpu434582536,23,4.4,12.914264,77.6784,12.924264,77.6884,2022-03-19,08:30:00,2026-08-30T08:45:00.000Z,Sandstorms,Low,motorcycle,Urban,130,Sports
rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,2026-08-30T18:10:00.000Z,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,2026-08-30T13:45:00.000Z,Cloudy,High,scooter,Metropolitian,150,Toys
fxuu788413734,22,4.8,17.431668,78.408321,17.461668,78.438321,2022-03-11,21:20:00,2026-08-30T21:30:00.000Z,Cloudy,Jam,motorcycle,Urban,130,Toys
njmo150975311,33,4.7,23.369746,85.33982,23.479746,85.44982,2022-03-04,19:15:00,2026-08-30T19:30:00.000Z,Fog,Jam,scooter,Metropolitian,200,Toys
jvjc772545076,35,4.6,12.352058,76.60665,12.482058,76.73665,2022-03-14,17:25:00,2026-08-30T17:30:00.000Z,Cloudy,Medium,motorcycle,Metropolitian,160,Snacks
uaeb808891380,22,4.8,17.433809,78.386744,17.563809,78.516744,2022-03-20,20:55:00,2026-08-30T21:05:00.000Z,Stormy,Jam,motorcycle,Metropolitian,170,Electronics
bgvc052754213,36,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55:00,2026-08-30T22:10:00.000Z,Fog,Jam,motorcycle,Metropolitian,230,Toys
